# Bronze Ingestion — TfL Arrivals

Ingest live London Underground arrivals for a configured set of monitored stations.

This notebook:

1. Loads monitored station configuration.
2. Resolves station names against the latest Silver StopPoint snapshot.
3. Calls the TfL Arrivals API for each station.
4. Validates each response.
5. Lands each original JSON response.
6. Appends each response to the Bronze Delta table.

**Target:** `workspace.urbanpulse_bronze.tfl_arrivals`

## 1. Initialise project paths

In [0]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parents[1]
SRC_PATH = PROJECT_ROOT / "src"

if not SRC_PATH.exists():
    raise FileNotFoundError(
        f"Source directory not found: {SRC_PATH}"
    )

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print(f"Project root: {PROJECT_ROOT}")

## 2. Import ingestion components


In [0]:
import uuid

from pyspark.sql import functions as F

from urbanpulse.ingestion.api_clients import ApiClient
from urbanpulse.ingestion.bronze import write_raw_bronze
from urbanpulse.ingestion.landing import land_json
from urbanpulse.utils.config import load_yaml

## 3. Load source and station configuration


In [0]:
SOURCE_CONFIG_PATH = (
    PROJECT_ROOT
    / "conf"
    / "sources.yml"
)

STATION_CONFIG_PATH = (
    PROJECT_ROOT
    / "conf"
    / "monitored_stations.yml"
)

source_config = load_yaml(
    str(SOURCE_CONFIG_PATH)
)

station_config = load_yaml(
    str(STATION_CONFIG_PATH)
)

tfl_config = source_config["tfl"]
arrivals_config = tfl_config["arrivals"]

BASE_URL = tfl_config["base_url"]
ENDPOINT_TEMPLATE = arrivals_config["endpoint"]

MONITORED_STATIONS = station_config["stations"]

BRONZE_TABLE = (
    "workspace."
    "urbanpulse_bronze."
    "tfl_arrivals"
)

STOP_POINTS_TABLE = (
    "workspace."
    "urbanpulse_silver."
    "tfl_stop_points"
)

LANDING_PATH = (
    "/Volumes/workspace/"
    "urbanpulse_meta/"
    "landing"
)

print(
    f"Configured stations: "
    f"{len(MONITORED_STATIONS)}"
)

## 4. Resolve monitored stations

Use the latest Silver StopPoint snapshot to resolve configured station names to stable TfL StopPoint IDs.

The pipeline fails if any configured station cannot be resolved uniquely.

In [0]:
stop_points_df = spark.table(
    STOP_POINTS_TABLE
)

latest_snapshot = (
    stop_points_df
    .agg(
        F.max("snapshot_at")
        .alias("latest_snapshot")
    )
    .first()["latest_snapshot"]
)

print(
    f"Latest reference snapshot: "
    f"{latest_snapshot}"
)

In [0]:
station_lookup_df = (
    stop_points_df
    .filter(
        F.col("snapshot_at") == latest_snapshot
    )
    .filter(
        F.col("common_name").isin(MONITORED_STATIONS)
    )
    .filter(
        F.col("station_naptan").isNotNull()
    )
    .select(
        F.col("station_naptan").alias("stop_point_id"),
        "common_name",
    )
    .dropDuplicates([
        "stop_point_id",
        "common_name",
    ])
)

display(
    station_lookup_df
    .orderBy("common_name")
)

## 5. Validate station resolution

Every configured station must resolve to exactly one TfL StopPoint ID before API calls begin.

In [0]:
from pyspark.sql import functions as F

resolution_check_df = (
    station_lookup_df
    .groupBy("common_name")
    .agg(
        F.countDistinct("stop_point_id").alias("id_count"),
        F.collect_set("stop_point_id").alias("stop_point_ids"),
    )
    .orderBy(F.col("id_count").desc(), "common_name")
)

display(resolution_check_df)

In [0]:
resolution_summary_df = (
    station_lookup_df
    .groupBy("common_name")
    .agg(
        F.countDistinct(
            "stop_point_id"
        ).alias("id_count")
    )
)

ambiguous_stations = (
    resolution_summary_df
    .filter(F.col("id_count") != 1)
)

if ambiguous_stations.count() > 0:
    display(ambiguous_stations)

    raise ValueError(
        "One or more monitored stations "
        "did not resolve to exactly one "
        "canonical station ID."
    )

resolved_stations = (
    station_lookup_df
    .collect()
)

resolved_names = {
    row["common_name"]
    for row in resolved_stations
}

missing_stations = (
    set(MONITORED_STATIONS)
    - resolved_names
)

if missing_stations:
    raise ValueError(
        "Unable to resolve configured stations: "
        f"{sorted(missing_stations)}"
    )

if len(resolved_stations) != len(
    MONITORED_STATIONS
):
    raise ValueError(
        "Resolved station count does not "
        "match configured station count."
    )

print(
    f"Successfully resolved "
    f"{len(resolved_stations)} stations."
)

display(
    station_lookup_df
    .orderBy("common_name")
)

## 6. Initialise the TfL API client

In [0]:
client = ApiClient(
    base_url=BASE_URL
)

## 7. Ingest arrivals for each monitored station

Each station request receives its own request ID, raw landing file, and Bronze record.

An empty arrivals list is accepted because it can represent a valid operational state.


In [0]:
successful_requests = []
failed_requests = []

for station in resolved_stations:

    station_id = station["stop_point_id"]
    station_name = station["common_name"]

    endpoint = (
        ENDPOINT_TEMPLATE
        .replace(
            "{station_id}",
            station_id,
        )
    )

    request_id = str(
        uuid.uuid4()
    )

    try:
        payload, status_code = client.get(
            endpoint=endpoint
        )

        if status_code != 200:
            raise RuntimeError(
                f"HTTP {status_code}"
            )

        if not isinstance(payload, list):
            raise TypeError(
                "Expected arrivals response "
                "to be a list"
            )

        landing_file = land_json(
            payload=payload,
            base_path=LANDING_PATH,
            source="tfl",
            dataset="arrivals",
            request_id=request_id,
        )

        write_raw_bronze(
            spark=spark,
            payload=payload,
            request_id=request_id,
            source="tfl",
            dataset="arrivals",
            source_endpoint=endpoint,
            http_status=status_code,
            table_name=BRONZE_TABLE,
        )

        successful_requests.append({
            "station_id": station_id,
            "station_name": station_name,
            "request_id": request_id,
            "arrivals": len(payload),
        })

        print(
            f"SUCCESS | "
            f"{station_name} | "
            f"{len(payload)} arrivals"
        )

    except Exception as exc:

        failed_requests.append({
            "station_id": station_id,
            "station_name": station_name,
            "error": str(exc),
        })

        print(
            f"FAILED | "
            f"{station_name} | "
            f"{exc}"
        )

## 8. Validate ingestion execution

All configured station API calls must succeed for this initial pipeline implementation.

In [0]:
print(
    f"Successful requests: "
    f"{len(successful_requests)}"
)

print(
    f"Failed requests: "
    f"{len(failed_requests)}"
)

if failed_requests:
    display(
        spark.createDataFrame(
            failed_requests
        )
    )

    raise RuntimeError(
        f"{len(failed_requests)} "
        "station requests failed."
    )

print(
    "All arrivals requests completed "
    "successfully."
)

## 9. Inspect the ingestion summary

In [0]:
summary_df = spark.createDataFrame(
    successful_requests
)

display(
    summary_df
    .orderBy("station_name")
)

## 10. Verify Bronze arrivals records

Each monitored station request is stored as an independent Bronze record.

In [0]:
%sql
SELECT
    request_id,
    source_endpoint,
    ingested_at,
    http_status,
    LENGTH(payload) AS payload_size
FROM workspace.urbanpulse_bronze.tfl_arrivals
ORDER BY ingested_at DESC;

In [0]:
successful_requests[0]

In [0]:
latest_arrival_payload = (
    spark.table(BRONZE_TABLE)
    .orderBy(
        F.col("ingested_at").desc()
    )
    .select("payload")
    .first()["payload"]
)

print(
    latest_arrival_payload[:2000]
)

In [0]:
display(
    dbutils.fs.ls(
        "/Volumes/workspace/"
        "urbanpulse_meta/"
        "landing/tfl/"
        "arrivals/"
    )
)